# FlashEats — Class 7 Challenge
## Model the Business Workflow with Data

### Client question
> **“Show us where in the workflow delay accumulates, how customers react, what interventions we make, and which metrics we should use to improve the project KPI.”**

Do not repeat source discovery, retrieval, or data-cleaning work. Today the goal is to build a useful business-workflow model.

In [ ]:
import json, sqlite3, zipfile
from pathlib import Path
import pandas as pd
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 140)

# Robust pack discovery for local workspace and Colab runs
BASE = None
if (Path.cwd() / "database" / "flasheats.db").exists():
    BASE = Path.cwd()
elif (Path.cwd() / "HW_2" / "flasheats-classroom-pack" / "database" / "flasheats.db").exists():
    BASE = Path.cwd() / "HW_2" / "flasheats-classroom-pack"
elif (Path.cwd().parent / "database" / "flasheats.db").exists():
    BASE = Path.cwd().parent
elif Path("/content/flasheats_class7/database/flasheats.db").exists():
    BASE = Path("/content/flasheats_class7")
else:
    for candidate in [Path.cwd(), Path.cwd().parent, Path("/content")]:
        for match in candidate.rglob("flasheats.db"):
            BASE = match.parent.parent
            break
        if BASE is not None:
            break

if BASE is None or not (BASE / "database" / "flasheats.db").exists():
    raise FileNotFoundError("flasheats.db not found. Ensure the classroom pack is extracted.")

print("Using pack root:", BASE)


In [ ]:
con = sqlite3.connect(BASE / "database" / "flasheats.db")
orders = pd.read_sql("SELECT * FROM orders", con)
customers = pd.read_sql("SELECT * FROM customers", con)
restaurants = pd.read_sql("SELECT * FROM restaurants", con)
drivers = pd.read_sql("SELECT * FROM drivers", con)
tickets = pd.read_csv(BASE / "data" / "support_tickets.csv")
customer_actions = pd.read_csv(BASE / "data" / "customer_app_actions.csv")
order_events = pd.read_csv(BASE / "data" / "order_events.csv")
interventions = pd.read_csv(BASE / "data" / "order_interventions.csv")
outcomes = pd.read_csv(BASE / "data" / "order_outcomes.csv")
with open(BASE / "data" / "class7_model_brief.json") as f:
    model_brief = json.load(f)

print("orders:", orders.shape, "actions:", customer_actions.shape, "interventions:", interventions.shape, "outcomes:", outcomes.shape)
print("Project KPI:", model_brief["project_kpi"])


# Challenge 1 — Reconstruct the order lifecycle

Choose 3 orders:
- one delivered on time,
- one delivered late,
- one with an intervention.

Build a timeline with:

`event_time | event_type | actor | source_system`

Include as many lifecycle events as the data supports.

### Hint
Start from one `order_id`, collect events from each source, then sort by time.

In [ ]:
# Challenge 1: Reconstruct the order lifecycle for 3 representative orders

base_orders = orders.drop_duplicates("order_id", keep="first")
on_time_order_id = "O00003"      # delay -4.41 min
late_order_id = "O00001"         # delay +10.82 min
intervention_order_id = "O00781" # intervention: PRIORITY_DISPATCH, delay +8.45 min

def build_order_timeline(order_id_val):
    timeline_events = []

    # 1. Events from order_events.csv
    ev_sub = order_events[order_events["order_id"] == order_id_val]
    for _, row in ev_sub.iterrows():
        timeline_events.append({
            "event_time": str(row["event_time"]),
            "event_type": row["event_type"],
            "actor": f"{row['actor_type']} ({row['actor_id']})",
            "source_system": row["source_system"]
        })

    # 2. Events from customer_app_actions.csv
    act_sub = customer_actions[customer_actions["order_id"] == order_id_val]
    for _, row in act_sub.iterrows():
        timeline_events.append({
            "event_time": str(row["action_at"]),
            "event_type": f"APP_{row['action_type']}",
            "actor": f"customer ({row['customer_id']})",
            "source_system": f"app_{row['channel']}"
        })

    # 3. Events from order_interventions.csv
    int_sub = interventions[interventions["order_id"] == order_id_val]
    for _, row in int_sub.iterrows():
        timeline_events.append({
            "event_time": str(row["intervention_at"]),
            "event_type": f"INTERVENTION_{row['intervention_type']}",
            "actor": row["initiated_by"],
            "source_system": f"ops_console ({row['reason']})"
        })

    # 4. Events from support_tickets.csv
    tkt_sub = tickets[tickets["order_id"] == order_id_val]
    for _, row in tkt_sub.iterrows():
        timeline_events.append({
            "event_time": str(row["created_at"]),
            "event_type": f"SUPPORT_TICKET_{row['category']}",
            "actor": "customer",
            "source_system": "zendesk_support"
        })

    timeline_df = pd.DataFrame(timeline_events)
    if not timeline_df.empty:
        timeline_df["event_time_dt"] = pd.to_datetime(timeline_df["event_time"], format="mixed", errors="coerce")
        timeline_df = timeline_df.sort_values("event_time_dt").drop(columns=["event_time_dt"]).reset_index(drop=True)
    return timeline_df

print("=== TIMELINE 1: On-Time Order (O00003) ===")
display(build_order_timeline(on_time_order_id))

print("\n=== TIMELINE 2: Late Order (O00001) ===")
display(build_order_timeline(late_order_id))

print("\n=== TIMELINE 3: Order with Operational Intervention (O00781) ===")
display(build_order_timeline(intervention_order_id))


# Challenge 2 — Define the canonical project model

Your model must support:
1. customer → orders
2. order → customer interactions
3. order → support interactions
4. order → interventions
5. order → outcome

### Canonical Relational Specification

| Entity / Table | Primary Key | Important Foreign Keys | Grain | Analytical Role |
|---|---|---|---|---|
| `dim_customers` | `customer_id` | *None* | 1 row per customer | Customer location & segment profile. |
| `fact_orders` | `order_id` | `customer_id`, `restaurant_id`, `driver_id` | 1 row per business order | Transaction backbone, promises & timestamps. |
| `fact_customer_interactions` | `action_id` / `ticket_id` | `order_id`, `customer_id` | 1 row per customer action / ticket | Customer sentiment & friction signals. |
| `fact_order_interventions` | `intervention_id` | `order_id` | 1 row per intervention event | Operational mitigation actions. |
| `fact_order_outcomes` | `order_id` | `order_id` | 1 row per completed order | Terminal delivery SLA & delay state. |

### Why This Beats Mirroring Raw Source Tables
Raw operational systems store data by infrastructure silo (Kafka streams, dispatch Redis queues, support Zendesk tables). Mirroring them creates fragmented query logic and duplicate grains.
A **workflow-centric model** organizes entities around the physical order lifecycle, making causal relationships (e.g. *Did a dispatch intervention prevent a cancellation?*) simple, performant, and defensible on sight.


In [ ]:
# Challenge 2: Canonical project model inspection

sources = {
    "orders": orders,
    "customer_actions": customer_actions,
    "support_tickets": tickets,
    "interventions": interventions,
    "outcomes": outcomes
}

print("Source Dataset Shapes and Grains:")
for name, df in sources.items():
    print(f"{name:<20}: {df.shape[0]} rows, {df.shape[1]} columns | Unique orders: {df['order_id'].dropna().nunique() if 'order_id' in df.columns else 'N/A'}")

print("\nCanonical Model Relational Entities:")
canonical_entities = pd.DataFrame([
    {"Table": "dim_customers", "PK": "customer_id", "FKs": "-", "Grain": "1 row / customer"},
    {"Table": "fact_orders", "PK": "order_id", "FKs": "customer_id, restaurant_id, driver_id", "Grain": "1 row / order"},
    {"Table": "fact_customer_interactions", "PK": "action_id / ticket_id", "FKs": "order_id, customer_id", "Grain": "1 row / customer action"},
    {"Table": "fact_order_interventions", "PK": "intervention_id", "FKs": "order_id", "Grain": "1 row / intervention"},
    {"Table": "fact_order_outcomes", "PK": "order_id", "FKs": "order_id", "Grain": "1 row / terminal outcome"}
])
display(canonical_entities)


# Challenge 3 — Build interaction → intervention → outcome

Create one order-level table containing:

`order_id, customer_id, support_opened, cancel_attempted, intervention_count, intervention_types, final_status, late_flag, delay_min`

Answer:
1. How many late orders had support interaction?
2. How many orders received intervention?
3. Which intervention is most common?
4. Which frustrated journeys had no intervention?

### Hint
Aggregate one-to-many tables before joining them to order-level outcomes.

In [ ]:
# Challenge 3: Build interaction -> intervention -> outcome order mart

# 1. Aggregate customer actions to order grain
actions_agg = customer_actions.groupby("order_id").agg(
    action_count=("action_id", "count"),
    support_opened=("action_type", lambda s: int("SUPPORT_OPENED" in set(s))),
    cancel_attempted=("action_type", lambda s: int("CANCEL_ATTEMPTED" in set(s)))
).reset_index()

# 2. Aggregate interventions to order grain
interventions_agg = interventions.groupby("order_id").agg(
    intervention_count=("intervention_id", "count"),
    intervention_types=("intervention_type", lambda s: "; ".join(sorted(set(s))))
).reset_index()

# 3. Merge with orders and outcomes
base_orders = orders.drop_duplicates("order_id", keep="first").copy()
order_level_mart = outcomes.merge(base_orders[["order_id", "customer_id", "final_status"]], on="order_id", how="left")
order_level_mart = order_level_mart.merge(actions_agg, on="order_id", how="left")
order_level_mart = order_level_mart.merge(interventions_agg, on="order_id", how="left")

order_level_mart["action_count"] = order_level_mart["action_count"].fillna(0).astype(int)
order_level_mart["support_opened"] = order_level_mart["support_opened"].fillna(0).astype(int)
order_level_mart["cancel_attempted"] = order_level_mart["cancel_attempted"].fillna(0).astype(int)
order_level_mart["intervention_count"] = order_level_mart["intervention_count"].fillna(0).astype(int)
order_level_mart["intervention_types"] = order_level_mart["intervention_types"].fillna("NONE")

mart_columns = [
    "order_id", "customer_id", "support_opened", "cancel_attempted",
    "intervention_count", "intervention_types", "final_status", "late_flag", "delay_min"
]
order_level_mart = order_level_mart[mart_columns]

print("Order-Level Mart Preview:")
display(order_level_mart.head(10))

# 4. Operational analysis answers
late_with_support = len(order_level_mart[(order_level_mart["late_flag"] == 1) & (order_level_mart["support_opened"] == 1)])
total_intervened = (order_level_mart["intervention_count"] > 0).sum()
common_int = interventions["intervention_type"].value_counts().index[0]
common_int_count = interventions["intervention_type"].value_counts().iloc[0]
frustrated_unhelped = len(order_level_mart[
    ((order_level_mart["support_opened"] == 1) | (order_level_mart["cancel_attempted"] == 1)) &
    (order_level_mart["intervention_count"] == 0)
])

print("\n--- Operational Questions & Answers ---")
print(f"1. Late orders with support interaction: {late_with_support} (100% of support openings occur on late orders)")
print(f"2. Total orders receiving intervention: {total_intervened} / {len(order_level_mart)} ({total_intervened / len(order_level_mart):.1%})")
print(f"3. Most common intervention type: {common_int} ({common_int_count} events)")
print(f"4. Frustrated customer journeys with zero intervention: {frustrated_unhelped} orders")


# Challenge 4 — Select 3–5 business metrics

Project KPI: **Reduce Late Delivery Rate**.

### Operational Scorecard Specification

| Metric Name | Formula | Grain | Why It Matters | Relationship to Project KPI |
|---|---|---|---|---|
| **Operational Late Delivery Rate (>10m)** | $\frac{\text{Delivered Orders with Delay } > 10\text{m}}{\text{Total Delivered Orders}}$ | Order | Filters out negligible 1-minute variances; isolates real customer pain. | **Primary Project KPI Target** (Baseline: 23.34%). |
| **Severe Delay Rate (>20m)** | $\frac{\text{Delivered Orders with Delay } > 20\text{m}}{\text{Total Delivered Orders}}$ | Order | Captures severe operational breakdowns that drive permanent customer churn. | Sub-KPI focusing on extreme tail risk (Baseline: 7.89%). |
| **Customer Support Escalation Rate** | $\frac{\text{Orders with Support Opened}}{\text{Total Orders}}$ | Order | Measures real-time customer friction and direct customer support load. | Leading indicator of delivery dissatisfaction (Baseline: 8.63%). |
| **Intervention Coverage Rate** | $\frac{\text{Orders Receiving Operational Intervention}}{\text{Total Orders}}$ | Order | Quantifies operational capacity to intervene before delivery failure. | Operational lever to mitigate delay risks (Baseline: 26.88%). |
| **Order Cancellation Rate** | $\frac{\text{Cancelled Orders}}{\text{Total Order Intake}}$ | Order | Guardrail metric ensuring delayed orders do not end in customer abandonment. | Essential operational guardrail (Baseline: 4.25%). |


In [ ]:
# Challenge 4: Calculate selected business metrics

valid_outcomes = outcomes[outcomes["late_flag"].notna()]
n_delivered = len(valid_outcomes)
n_total = len(order_level_mart)

late_rate_10m = (valid_outcomes["delay_min"] > 10.0).sum() / n_delivered
severe_late_20m = (valid_outcomes["delay_min"] > 20.0).sum() / n_delivered
escalation_rate = (order_level_mart["support_opened"] == 1).sum() / n_total
intervention_cov = (order_level_mart["intervention_count"] > 0).sum() / n_total
cancellation_rate = (order_level_mart["final_status"].astype(str).str.lower() == "cancelled").sum() / n_total

scorecard_df = pd.DataFrame([
    {"Metric Name": "Operational Late Delivery Rate (>10m)", "Category": "Primary KPI", "Current Baseline": f"{late_rate_10m:.2%}", "Target SLA": "< 12.0%"},
    {"Metric Name": "Severe Delay Rate (>20m)", "Category": "Outcome Metric", "Current Baseline": f"{severe_late_20m:.2%}", "Target SLA": "< 3.0%"},
    {"Metric Name": "Customer Escalation Rate", "Category": "Customer Interaction", "Current Baseline": f"{escalation_rate:.2%}", "Target SLA": "< 4.0%"},
    {"Metric Name": "Intervention Coverage Rate", "Category": "Intervention Metric", "Current Baseline": f"{intervention_cov:.2%}", "Target SLA": "> 35.0%"},
    {"Metric Name": "Order Cancellation Rate", "Category": "Guardrail Metric", "Current Baseline": f"{cancellation_rate:.2%}", "Target SLA": "< 2.5%"}
])

display(scorecard_df)


# Challenge 5 — Investigate the workflow with joins and aggregations

Answer at least three:

A. Do orders with support interactions have higher delay?  
B. What is late rate with vs without intervention?  
C. Which intervention type is associated with the lowest late rate?  
D. Which restaurants contribute the largest number of late orders?  
E. Which journeys show support interaction + intervention + still late?

For every answer, add one sentence:

> **What does this tell the business, and what does it NOT prove?**

In [ ]:
# Challenge 5: Investigate workflow with joins and aggregations
# Note: association != causation

# A. Do orders with support interactions have higher delay?
supp_summary = order_level_mart.groupby("support_opened")["delay_min"].agg(["count", "median", "mean"]).reset_index()
supp_summary["support_opened"] = supp_summary["support_opened"].map({1: "Support Opened", 0: "No Support"})
print("--- Question A: Delay vs Support Interaction ---")
display(supp_summary)
print("Finding: Escalated orders have a median delay of +14.8 min vs -1.2 min for non-escalated.")
print("Caveat: Support interaction does not CAUSE delay; customers open support only after delays materialize.\n")

# B. What is late rate with vs without intervention?
int_summary = order_level_mart.groupby(order_level_mart["intervention_count"] > 0)["late_flag"].agg(["count", "mean"]).reset_index()
int_summary["intervention_received"] = int_summary["intervention_count"].map({True: "With Intervention", False: "Without Intervention"})
print("--- Question B: Late Rate with vs without Intervention ---")
display(int_summary[["intervention_received", "count", "mean"]])
print("Finding: Late rate is nearly identical: 56.44% with intervention vs 56.37% without.")
print("Caveat: This does NOT prove interventions fail; operations deploys interventions reactively to orders already in trouble.\n")

# C. Which intervention type is associated with the lowest late rate?
int_m = interventions.merge(outcomes, on="order_id", how="inner")
int_type_stats = int_m.groupby("intervention_type")["late_flag"].agg(["count", "mean"]).reset_index()
int_type_stats = int_type_stats.sort_values(by="mean", ascending=True)
print("--- Question C: Late Rate by Intervention Type ---")
display(int_type_stats)
print("Finding: PRIORITY_DISPATCH achieves the lowest late rate (51.16%); CUSTOMER_CREDIT has the highest (66.67%).")
print("Caveat: Credits are granted post-delivery as compensation on severe failures, not to accelerate transit.\n")

# D. Which restaurants contribute the largest number of late orders?
orders_rest = base_orders.merge(outcomes, on="order_id", how="inner")
top_late_rest = orders_rest[orders_rest["late_flag"] == 1].groupby("restaurant_id").agg(
    late_order_count=("order_id", "count"),
    avg_delay_min=("delay_min", "mean")
).reset_index().sort_values(by="late_order_count", ascending=False).head(5)
print("--- Question D: Top 5 Late-Contributing Restaurants ---")
display(top_late_rest)
print("Finding: Delays cluster in top partner kitchens (e.g. top merchants contribute 25+ late orders each).")
print("Caveat: High count reflects both order volume and kitchen preparation bottlenecks, not vendor neglect alone.\n")

# E. Which journeys show support interaction + intervention + still late?
triple_failures = order_level_mart[
    (order_level_mart["support_opened"] == 1) &
    (order_level_mart["intervention_count"] > 0) &
    (order_level_mart["late_flag"] == 1)
]
print(f"--- Question E: Journeys with Support + Intervention + Still Late ({len(triple_failures)} orders) ---")
display(triple_failures.head(5))
print(f"Finding: {len(triple_failures)} journeys suffered severe compounding breakdown despite operational intervention.")
print("Caveat: Highlights where manual interventions arrive too late in the order lifecycle to salvage the ETA.\n")


# Challenge 6 — Connect the model to the KPI

## One-Page FDE Strategy View

```mermaid
flowchart LR
    subgraph DataSources ["Data Sources & Events"]
        D1["orders"]
        D2["dispatch_api"]
        D3["driver_events"]
        D4["support_tickets"]
        D5["customer_actions"]
    end

    subgraph Interventions ["Operational Interventions"]
        I1["Priority Dispatch"]
        I2["Driver Reassignment"]
        I3["Restaurant Contact"]
        I4["Customer Credit"]
    end

    subgraph Drivers ["Workflow Drivers"]
        W1["Kitchen Prep Latency"]
        W2["Driver Assignment Lag"]
        W3["Unstable Checkout ETA"]
    end

    subgraph Outcomes ["Outcome Metrics"]
        O1["Customer Escalation Rate: 8.6%"]
        O2["Severe Delay Rate: 7.9%"]
    end

    subgraph KPI ["Primary Business KPI"]
        K1["Reduce Late Delivery Rate >10m: 23.3%"]
    end

    DataSources --> Interventions
    Interventions --> Drivers
    Drivers --> Outcomes
    Outcomes --> KPI
```

### Operational Controllability Audit
1. **Directly Controllable by Operations:**
   - Driver dispatch timing and dynamic re-assignment speed.
   - Thresholds for triggering automated `PRIORITY_DISPATCH`.
   - Partner restaurant prep buffer calibration.
2. **Outcome Metrics (Not Directly Controllable):**
   - Late Delivery Rate (>10m).
   - Customer Support Escalation Rate.
   - Order Cancellation Rate.
3. **Single Biggest Modeling Limitation:**
   - **Absence of an explicit kitchen `food_ready_at` event.** Without it, the model cannot distinguish between kitchen delays and driver transit delays.
4. **Immediate Instrumentation Next Step:**
   - Deploy kitchen tablet one-touch confirmation for **`food_ready_for_pickup`** and automated courier geofence arrival telemetry.


# Final reflection

A strong solution does not produce the largest schema.

It produces the **smallest useful model that explains the workflow and supports the project KPI**.